In [ ]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
from tensorflow import keras
import keras

import tensorflow as tf
import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras import optimizers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.utils import plot_model


In [ ]:
import joblib

scaler = joblib.load("/content/drive/MyDrive/Colab Notebooks/nids/preprocessing_tools/scaler.pkl")

FEATURE_COLUMNS = None    # will lock after first dataset
LABEL_COLUMN = "Label"

In [ ]:
def load_and_preprocess(path):
    global FEATURE_COLUMNS

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    # Drop missing label safe-case
    if LABEL_COLUMN in df.columns:
        labels = df[LABEL_COLUMN]
        df = df.drop(columns=[LABEL_COLUMN])
    else:
        labels = None

    # Lock feature list from first file
    if FEATURE_COLUMNS is None:
        FEATURE_COLUMNS = df.columns.tolist()
        print("Locked Features:", len(FEATURE_COLUMNS))
    else:
        df = df[FEATURE_COLUMNS]  # reorder / ensure consistency

    # Clean data
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(df.median(numeric_only=True))

    # Scale with TRAIN scaler
    X = scaler.transform(df.values)

    return X.astype("float32"), labels

In [ ]:
# ------------------------------
# CONFIG
# ------------------------------
MODELS_DIR = "/content/drive/MyDrive/Colab Notebooks/nids/models/cnnae-again"
BENIGN_TEST_PATH = "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Monday-WorkingHours.pcap_ISCX.csv"

ATTACK_FILES = [
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Wednesday-workingHours.pcap_ISCX.csv",
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Tuesday-WorkingHours.pcap_ISCX.csv",
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)


# ------------------------------
# HELPERS
# ------------------------------
def load_csv(path):
    df = pd.read_csv(path)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df.values.astype("float32")


def mse(a, b):
    return np.mean(np.power(a - b, 2), axis=1)


def compute_threshold(errors):
    return errors.mean() + 3 * errors.std()      # Gaussian rule
    # return np.percentile(errors, 99)           # Alternative


def evaluate_model(model, X_benign, attack_sets):

    # --- benign prediction ---
    recon_benign = model.predict(X_benign, verbose=0)
    benign_errors = mse(X_benign, recon_benign)

    threshold = compute_threshold(benign_errors)

    results = []

    for attack_name, X_attack in attack_sets.items():

        # predict attack
        recon_attack = model.predict(X_attack, verbose=0)
        attack_errors = mse(X_attack, recon_attack)

        # labels
        y_true = np.concatenate([np.zeros(len(X_benign)), np.ones(len(X_attack))])
        y_scores = np.concatenate([benign_errors, attack_errors])

        y_pred = (y_scores > threshold).astype(int)

        # metrics
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )

        auc = roc_auc_score(y_true, y_scores)

        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()

        acc = (tp + tn) / (tp + tn + fp + fn)
        fpr = fp / (fp + tn + 1e-9)

        results.append({
            "Attack": attack_name,
            "Threshold": threshold,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "AUC": auc,
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn
        })

    return pd.DataFrame(results)

In [ ]:
# ------------------------------
# MAIN
# ------------------------------
print("Loading benign test data...")
X_benign_test, _ = load_and_preprocess(BENIGN_TEST_PATH)

print("Loading attack datasets...")
attack_sets = {}
for f in ATTACK_FILES:
    X, labels = load_and_preprocess(f)
    attack_sets[os.path.basename(f)] = X

Loading benign test data...
Locked Features: 78


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Loading attack datasets...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/valida

In [ ]:
@keras.saving.register_keras_serializable()
class Encoder(tf.keras.Model):
    def __init__(self, input_shape, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_ = input_shape
        self.latent_dim = latent_dim

        self.conv1 = layers.Conv1D(32, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv1D(latent_dim, 3, padding="same", activation="relu")
        self.maxpool = layers.MaxPooling1D(2, padding="same")

    def call(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return self.maxpool(x)

    def get_config(self):
        return {
            "input_shape": self.input_shape_,
            "latent_dim": self.latent_dim,
        }

@keras.saving.register_keras_serializable()
class Decoder(tf.keras.Model):
    def __init__(self, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim

        self.deconv1 = layers.Conv1D(latent_dim, 3, padding="same", activation="relu")
        self.upsample = layers.UpSampling1D(2)
        self.output_layer = layers.Conv1D(1, 3, padding="same", activation="linear")

    def call(self, x):
        x = self.deconv1(x)
        x = self.upsample(x)
        return self.output_layer(x)

    def get_config(self):
        return {
            "latent_dim": self.latent_dim,
        }

@keras.saving.register_keras_serializable()
class ConvAutoencoder(tf.keras.Model):
    def __init__(self, input_shape, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_ = input_shape
        self.latent_dim = latent_dim

        self.encoder = Encoder(input_shape, latent_dim)
        self.decoder = Decoder(latent_dim)

    def call(self, x):
        return self.decoder(self.encoder(x))

    def get_config(self):
        return {
            "input_shape": self.input_shape_,
            "latent_dim": self.latent_dim,
        }

In [ ]:
summary_rows = []

for model_file in os.listdir(MODELS_DIR):
    if not model_file.endswith(".keras"):
        continue

    model_path = os.path.join(MODELS_DIR, model_file)
    print(f"\nEvaluating model: {model_file}")

    model = load_model(model_path, compile=False)

    df_results = evaluate_model(model, X_benign_test, attack_sets)

    save_path = os.path.join(RESULTS_DIR, f"{model_file}_evaluation.csv")
    df_results.to_csv(save_path, index=False)

    print(df_results)

    # Store mean performance for final ranking
    summary_rows.append({
        "Model": model_file,
        "Mean Recall": df_results["Recall"].mean(),
        "Mean F1": df_results["F1"].mean(),
        "Mean AUC": df_results["AUC"].mean(),
        "Mean Accuracy": df_results["Accuracy"].mean(),
        "Worst Recall": df_results["Recall"].min()   # important robustness metric
    })


summary = pd.DataFrame(summary_rows)
summary = summary.sort_values(by=["Mean Recall", "Mean F1", "Mean AUC"], ascending=False)

summary_path = os.path.join(RESULTS_DIR, "MODEL_COMPARISON_SUMMARY.csv")
summary.to_csv(summary_path, index=False)

print("\n\n================ FINAL MODEL RANKING ================")
print(summary)


Evaluating model: cnnae_base.keras


ValueError: Exception encountered when calling Encoder.call().

[1mInput 0 of layer "conv1d" is incompatible with the layer: expected min_ndim=3, found ndim=2. Full shape received: (32, 78)[0m

Arguments received by Encoder.call():
  • x=tf.Tensor(shape=(32, 78), dtype=float32)

In [ ]:
!zip -r results.zip ./results